# Automated Resume Shortlisting — Updated NLP Pipeline
### Changes: LLM-based extraction (strict prompt), validation layer, experience breakdown, removed hardcoded skill maps & regex experience

## Install Dependencies

In [107]:
# 1. Force NumPy < 2.0 (CRITICAL for scispaCy compatibility on Kaggle)
#!pip install "numpy<2.0" --force-reinstall

# 2. Install spaCy and aligned scispaCy version
#!pip install spacy==3.7.4 scispacy==0.5.5
#!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.5/en_core_sci_sm-0.5.5.tar.gz

# 3. Install other essential libraries
#!pip install pdfplumber python-docx openai sentence-transformers torch transformers

print("\n✅ Dependencies installed. PLEASE RESTART KERNEL NOW!")


✅ Dependencies installed. PLEASE RESTART KERNEL NOW!


## All Imports

In [108]:
import numpy as np
import os
import re
import json
import shutil
import docx
import spacy
import pdfplumber
import openai
from pathlib import Path

from sentence_transformers import SentenceTransformer, CrossEncoder, util
from scispacy.abbreviation import AbbreviationDetector

print("✅ All essential imports successful.")

✅ All essential imports successful.


In [109]:
import math
import numpy as np

def normalize_degree(deg):
    if not deg: return ""
    deg = str(deg).lower()
    if any(x in deg for x in ["btech", "b.tech", "bachelor", "b.e.", "be"]): return "bachelor"
    if any(x in deg for x in ["mtech", "m.tech", "master", "ms", "m.e.", "me"]): return "master"
    if any(x in deg for x in ["phd", "ph.d", "doctorate"]): return "doctorate"
    return deg

def normalize_skill(skill):
    """Minimal normalization, relying on LLM for semantic mapping."""
    skill = str(skill).lower().strip()
    synonyms = {
        'ml': 'machine learning', 'ai': 'artificial intelligence', 'nlp': 'natural language processing',
        'js': 'javascript', 'ts': 'typescript', 'py': 'python'
    }
    return synonyms.get(skill, skill)

def compute_semantic_similarity(jd_text, resume_text):
    jd_vec = embedding_model.encode(jd_text, convert_to_tensor=True)
    res_vec = embedding_model.encode(resume_text, convert_to_tensor=True)
    sim = util.cos_sim(jd_vec, res_vec).item()
    return float(max(0.0, sim))

def compute_reranker_score(jd_text, resume_text):
    """Better separation for reranker scores, handling 512 token limit."""
    score = reranker_model.predict([jd_text[:1200], resume_text[:1200]])
    return float(1 / (1 + math.exp(-score / 1.5)))

def compute_evidence_score(matched_skills, exp_text, projects):
    if not matched_skills: return 0.0
    text = (str(exp_text) * 2 + " " + " ".join(map(str, projects))).lower()
    count = sum(1 for skill in matched_skills if normalize_skill(skill) in text)
    return float(count / len(matched_skills))


## Import Dataset from Kaggle

In [110]:
import os
KAGGLE = os.path.exists("/kaggle")
dataset_path = "/kaggle/input/datasets/kartavayapatel/resume-new-set" if KAGGLE else "./raw_resumes"

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/kartavayapatel/resume-new-set/MT2025065 - Keyur Padiya.pdf
/kaggle/input/datasets/kartavayapatel/resume-new-set/MT2024065_SDE_RESUME.pdf
/kaggle/input/datasets/kartavayapatel/resume-new-set/MT2025047_GauravRajpurohit.pdf
/kaggle/input/datasets/kartavayapatel/resume-new-set/MT2025059 - Kartavyakumar Patel.pdf
/kaggle/input/datasets/kartavayapatel/resume-new-set/MT2025013 - Aditya Dave.pdf
/kaggle/input/datasets/kartavayapatel/resume-new-set/MT2025069 (2).pdf
/kaggle/input/datasets/kartavayapatel/resume-new-set/MT2025085.pdf
/kaggle/input/datasets/kartavayapatel/resume-new-set/JD.pdf


## Rename Resume Files to Meaningful Names

In [111]:
import os
KAGGLE = os.path.exists("/kaggle")
input_path = "/kaggle/input/datasets/kartavayapatel/resume-new-set" if KAGGLE else "./raw_resumes"
output_path = "/kaggle/working/dataset" if KAGGLE else "./dataset"

os.makedirs(output_path, exist_ok=True)

resume_count = 1

for root, dirs, files in os.walk(input_path):
    for file in sorted(files):
        old_path = os.path.join(root, file)

        if "jd" in file.lower():
            new_name = "JD.pdf"
        else:
            new_name = f"resume_{resume_count}.pdf"
            resume_count += 1

        new_path = os.path.join(output_path, new_name)
        shutil.copy(old_path, new_path)

print("Dataset prepared successfully!")

Dataset prepared successfully!


## Section 1: Text Extraction

In [112]:
def extract_text_from_pdf(file):
    text = ""
    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            content = page.extract_text()
            if content:
                text += content + "\n"
    return text


def extract_text_from_docx(file_path):
    doc = docx.Document(file_path)
    return "\n".join([para.text for para in doc.paragraphs])


def extract_text(file_path):
    if file_path.lower().endswith(".pdf"):
        return extract_text_from_pdf(file_path)
    elif file_path.lower().endswith(".docx"):
        return extract_text_from_docx(file_path)
    return ""

## Section 2: Basic Cleaning

In [113]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s\+\#\.]', ' ', text)  # Keep tech chars like C++, Node.js
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def fix_numbers(text):
    """Fix OCR-mangled numbers: 3 72 → 3.72"""
    text = re.sub(r'\b([0-4])\s+([0-9]{2})\b', r'\1.\2', text)
    text = re.sub(r'\b([0-9])\s+00\b', r'\1.00', text)
    return text

## Process JD and Resumes

In [114]:
import os
KAGGLE = os.path.exists("/kaggle")
dataset_path = "/kaggle/working/dataset" if KAGGLE else "./dataset"

jd_text = ""
resume_text_list = []

for file in sorted(os.listdir(dataset_path)):
    file_path = os.path.join(dataset_path, file)

    if not os.path.isfile(file_path):
        continue

    raw_text = extract_text(file_path)
    cleaned_text = clean_text(raw_text)
    cleaned_text = fix_numbers(cleaned_text)

    if file.lower() == "jd.pdf":
        jd_text = cleaned_text
    else:
        resume_text_list.append(cleaned_text)

print("JD Length:", len(jd_text))
print("Number of Resumes:", len(resume_text_list))

JD Length: 2009
Number of Resumes: 7


## Section 3: Text Normalization (scispaCy abbreviation expansion — light only)

In [115]:
# ✅ KEPT: Light normalization with scispaCy
try:
    nlp = spacy.load("en_core_sci_sm")
    nlp.add_pipe("abbreviation_detector")
    print("✅ scispaCy loaded successfully")
except OSError:
    print("⚠️  Falling back to regular spaCy")
    nlp = spacy.load("en_core_web_sm")

# Minimal fallback abbreviations — DO NOT expand into full skill lists
fallback_abbreviations = {
    "ml": "machine learning",
    "ai": "artificial intelligence",
    "nlp": "natural language processing",
    "dl": "deep learning",
    "cv": "computer vision"
}


def normalize_text(text):
    """Light normalization: abbreviation expansion + lemmatization. No skill extraction here."""
    doc = nlp(text)
    expanded_text = text.lower()

    if hasattr(doc._, 'abbreviations'):
        for abrv in doc._.abbreviations:
            short_form = str(abrv).lower()
            long_form = str(abrv._.long_form).lower()
            expanded_text = re.sub(
                r'\b' + re.escape(short_form) + r'\b', long_form, expanded_text
            )

    for abbr, full in fallback_abbreviations.items():
        expanded_text = re.sub(
            r'\b' + re.escape(abbr) + r'\b', full, expanded_text
        )

    doc = nlp(expanded_text)
    tokens = []
    i = 0
    while i < len(doc):
        token = doc[i]
        if (
            i < len(doc) - 2
            and doc[i].like_num
            and doc[i + 1].text == "."
            and doc[i + 2].like_num
        ):
            tokens.append(doc[i].text + "." + doc[i + 2].text)
            i += 3
            continue
        if not token.is_stop and not token.is_punct:
            tokens.append(token.lemma_)
        i += 1

    return " ".join(tokens)

⚠️  Falling back to regular spaCy


In [116]:
clean_jd_text = normalize_text(jd_text)
clean_resume_text = [normalize_text(r) for r in resume_text_list]

print("\nJD Preview:\n", clean_jd_text[:2000])


JD Preview:
 role mts intern member technical staff job summary seek highly motivated talented mts intern join team intern responsible web development automation programming task work variety project collaborate cross functional team deliver high quality software solution excellent opportunity gain practical experience enhance skill fast pace environment join impact netapp job requirement good programming skill c c++ python java golang understanding datum structure algorithm basic multi threading concept exposure nodejs javascript typescript web development frontend backend familiarity linux basic operating system fundamental experience python scripting automation simple testing framework like pyt basic understanding coursework exposure docker kubernete ci cd tool prefer require knowledge rest apis microservice basic distribute system concept plus familiarity cloud platform openstack cloudstack cloud technology nice interest datum science machine learning artificial intelligence tool 

## Section 4: LLM-Based Information Extraction (INDUSTRY-LEVEL)



### Uses:
- LLM with strict prompt to extract skills, experience, education, projects
- Experience breakdown (role + type + duration) to avoid BTech/project confusion
- Post-extraction validation layer

In [117]:
import os
import re
os.environ["OPENROUTER_API_KEY"] = "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"
client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

def build_extraction_prompt(text, is_jd=False):
    role = "expert resume parser" if not is_jd else "expert job description analyzer"
    subject = "resume" if not is_jd else "job description"
    exp_rule = "Include ONLY professional work experience entries." if not is_jd else "Include minimum required years of experience."
    
    name_extraction = ""
    if not is_jd:
        name_extraction = '6. "candidate_name": Extract the full name of the candidate. Exclude institute names, locations, or degree titles.'

    return f'''
You are an {role}.

Analyze the {subject} text provided below and extract both semantic descriptions and structured data.

STRICT RULES FOR EXTRACTION:
1. "experience": {exp_rule}
   - Include: Full-time jobs, internships (with role, type, duration_years).
   - EXCLUDE: Education (BTech, MTech, degrees), academic projects, training courses.
   - Only count actual job/internship roles as experience.

2. "skills": Extract ALL technical skills, tools, and programming languages found ANYWHERE in the text (including deep inside project descriptions, experience bullets, and education). Do not miss any framework or language.

3. "education": Extract degree names only (BTech, MTech, etc.).

4. "projects": Include project titles or short descriptions (1 sentence).

5. "semantic_summary": Provide a 2-3 sentence professional summary focusing on candidate's technical profile.

{name_extraction}

OUTPUT FORMAT (STRICT JSON ONLY):
{{
  "candidate_name": "full name",
  "skills": ["list of strings"],
  "experience": [
    {{
      "role": "title",
      "type": "internship/full-time",
      "duration_years": number
    }}
  ],
  "education": ["degrees"],
  "projects": ["descriptions"],
  "semantic_summary": "summary text"
}}

{subject.capitalize()}:
{text}
'''

def rule_based_fallback_extraction(text, is_jd=False):
    tech_keywords = ['python', 'java', 'c++', 'c', 'javascript', 'typescript', 'go', 'ruby', 'react', 'node.js', 'angular', 'sql', 'mysql', 'postgres', 'mongodb', 'aws', 'docker', 'kubernetes', 'linux', 'unix', 'machine learning', 'deep learning', 'nlp', 'computer vision', 'html', 'css', 'spring', 'django', 'flask', 'fastapi']
    text_lower = text.lower()
    found_skills = [skill for skill in tech_keywords if skill in text_lower or f" {skill} " in text_lower]
    
    years = 0
    exp_matches = re.findall(r'(\d+)\+?\s*(?:years?|yrs?)(?:\s+of)?\s+experience', text_lower)
    if exp_matches:
        try:
            years = max([int(m) for m in exp_matches])
        except: pass
            
    name = ""
    if not is_jd:
        try:
            doc = nlp(text[:500])
            for ent in doc.ents:
                if ent.label_ == "PERSON":
                    name = ent.text
                    break
        except: pass
                
    return {
        "candidate_name": name,
        "skills": list(set(found_skills)),
        "experience": [{"role": "unspecified", "type": "full-time", "duration_years": years}] if years > 0 else [],
        "education": [],
        "projects": [],
        "semantic_summary": "Auto-generated by rule-based fallback due to LLM extraction failure."
    }

def call_llm_extraction(text, is_jd=False, retries=2):
    prompt = build_extraction_prompt(text[:4000], is_jd=is_jd)
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="openai/gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"}
            )
            raw = response.choices[0].message.content.strip()
            raw = re.sub(r'^```json\s*|^```\s*|```$', '', raw, flags=re.MULTILINE).strip()
            return json.loads(raw)
        except Exception as e:
            if attempt == retries - 1:
                print(f"LLM Error after {retries} attempts: {e}")
                return rule_based_fallback_extraction(text, is_jd)
            print(f"LLM parsing failed, retrying... ({attempt+1}/{retries})")


In [119]:
import re

def validate_experience(exp):
    if exp > 50: return 50.0
    if exp < 0: return 0.0
    return float(exp)

def fix_common_errors(resume_text, total_exp):
    return total_exp

def parse_duration(d):
    if isinstance(d, (int, float)): return float(d)
    if isinstance(d, str):
        match = re.search(r'(\d+(?:\.\d+)?)', d)
        if match: return float(match.group(1))
    return 0.0

def compute_experience_years(experience_list, resume_text, is_jd=False):
    total = sum(parse_duration(e.get("duration_years", 0)) for e in experience_list if isinstance(e, dict))
    if total == 0:
        text_lower = resume_text.lower()
        exp_matches = re.findall(r'(\d+)\+?\s*(?:years?|yrs?)(?:\s+of)?\s+experience', text_lower)
        if exp_matches:
            try: total = float(max([int(m) for m in exp_matches]))
            except: pass
            
    if not is_jd:
        total = fix_common_errors(resume_text, total)
    return validate_experience(total)

def validate_extracted_struct(struct, resume_text, is_jd=False):
    name = struct.get("candidate_name", "").strip()
    skills = [s.strip() for s in struct.get("skills", []) if isinstance(s, str)]
    edu = [e.strip() for e in struct.get("education", []) if isinstance(e, str)]
    proj = [p.strip() for p in struct.get("projects", []) if isinstance(p, str)]
    exp_entries = struct.get("experience", [])
    
    return {
        "name": name,
        "skills": skills,
        "experience_years": compute_experience_years(exp_entries, resume_text, is_jd=is_jd),
        "experience_breakdown": exp_entries,
        "education": edu,
        "projects": proj,
        "semantic_summary": struct.get("semantic_summary", "")
    }


In [120]:
# ============================================================
# 🔷 EXTRACT ALL RESUMES + JD via LLM
# ============================================================

print("🔍 Extracting structured data from resumes via LLM...")
print("=" * 60)

Resume_struct = []

for i, resume_text in enumerate(resume_text_list):
    print(f"  Processing Resume {i+1}/{len(resume_text_list)}...")
    raw_struct = call_llm_extraction(resume_text, is_jd=False)
    validated = validate_extracted_struct(raw_struct, resume_text, is_jd=False)
    Resume_struct.append(validated)

print("\n📋 Extracting JD structure...")
raw_jd_struct = call_llm_extraction(jd_text, is_jd=True)
JD_struct = validate_extracted_struct(raw_jd_struct, jd_text, is_jd=True)

print("\n✅ Extraction complete!")
print(f"   JD Skills: {JD_struct['skills'][:8]}")
print(f"   JD Education: {JD_struct['education']}")
print(f"   JD Experience Req: {JD_struct['experience_years']} years")

🔍 Extracting structured data from resumes via LLM...
  Processing Resume 1/7...
  Processing Resume 2/7...
  Processing Resume 3/7...
  Processing Resume 4/7...
  Processing Resume 5/7...
  Processing Resume 6/7...
  Processing Resume 7/7...

📋 Extracting JD structure...

✅ Extraction complete!
   JD Skills: ['C', 'C++', 'Python', 'Java', 'Go', 'Data Structures', 'Algorithms', 'Multithreading']
   JD Education: ["Master's degree"]
   JD Experience Req: 0.0 years


In [121]:
# Preview extracted resume structures
for i, struct in enumerate(Resume_struct[:3]):
    print(f"\n📄 Resume {i+1}")
    print("-" * 40)
    print(f"Skills      : {', '.join(struct['skills'][:8])}")
    print(f"Education   : {', '.join(struct['education'])}")
    print(f"Exp (years) : {struct['experience_years']}")
    print(f"Exp Detail  : {struct['experience_breakdown']}")
    print(f"Projects    : {struct['projects'][:3]}")


📄 Resume 1
----------------------------------------
Skills      : C, C++, SQL, HTML, CSS, JavaScript, React.js, Node.js
Education   : MTech, BTech
Exp (years) : 1.0
Exp Detail  : [{'role': 'System Software Engineer Intern', 'type': 'internship', 'duration_years': 1}]
Projects    : ['Developed HostelHub, a microservices-based hostel management system enabling real-time room allocation and deallocation.', 'Developed a data synchronization system integrating MySQL, MongoDB, and Hive to ensure 100% consistency across 5,000+ student grade records.', 'Designed and implemented an e-book platform for merging multiple sources to create custom books.']

📄 Resume 2
----------------------------------------
Skills      : C++, Python, JavaScript, SQL, React.js, Tailwind CSS, Spring Boot, Pandas
Education   : MTech, BTech
Exp (years) : 0.5
Exp Detail  : [{'role': 'App Developer', 'type': 'internship', 'duration_years': 0.5}]
Projects    : ['Designed an end to end data warehouse using the medallion b

## Section 5: Semantic Embeddings

In [122]:
from sentence_transformers import SentenceTransformer, CrossEncoder

# Define model names
EMBEDDING_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Initialize models
print("⏳ Loading NLP Models (this may take a minute)...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
reranker_model = CrossEncoder(RERANKER_MODEL_NAME)
print("✅ Models loaded successfully.")


⏳ Loading NLP Models (this may take a minute)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Models loaded successfully.


In [123]:
# --- CANDIDATE NAME INITIALIZATION ---

def extract_candidate_name_from_resume_text(resume_text):
    """Strict top-line name extraction to avoid grabbing institute names."""
    lines = [l.strip() for l in resume_text.split('\n') if l.strip()]
    if not lines: return None
    
    # Take the first line and limit to first 4 words
    first_line = lines[0]
    words = [w for w in first_line.split() if w.isalpha() and len(w) > 1]
    
    if len(words) >= 2:
        return " ".join(words[:4]).title()
        
    email_match = re.search(r"([a-zA-Z]+)[._]([a-zA-Z]+)@", resume_text)
    if email_match:
        return f"{email_match.group(1).title()} {email_match.group(2).title()}"
    return None


def build_candidate_names(dataset_path, resume_text_list):
    resume_files = [
        f for f in sorted(os.listdir(dataset_path))
        if os.path.isfile(os.path.join(dataset_path, f)) and f.lower() != "jd.pdf"
    ] if dataset_path and os.path.isdir(dataset_path) else []

    names = []
    for idx, resume_text in enumerate(resume_text_list):
        fn_name = normalize_candidate_name_from_filename(resume_files[idx]) if idx < len(resume_files) else None
        rt_name = extract_candidate_name_from_resume_text(resume_text)
        final = fn_name if fn_name else (rt_name if rt_name else f"Candidate {idx + 1}")
        names.append(final)
    return names

def normalize_candidate_name_from_filename(filename):
    stem = Path(filename).stem
    stem = re.sub(r"\(\d+\)$", "", stem).strip()
    stem = stem.replace("_", " ").replace("-", " ")
    stem = re.sub(r"\b(mt|bt|cv|resume|updated|final|copy|sde)\b", " ", stem, flags=re.IGNORECASE)
    stem = re.sub(r"\s+", " ", stem).strip()
    words = [w for w in stem.split() if w.isalpha()]
    return " ".join(words[:4]).title() if len(words) >= 2 else None

candidate_names = build_candidate_names(dataset_path, resume_text_list)
print("Candidate Names Prepared:", candidate_names)


Candidate Names Prepared: ['Jainish Parmar International Institute', 'Aditya Dave International Institute', 'Gaurav Rajpurohit International Institute', 'Kartavya Patel International Institute', 'Keyur Sanjaykumar Padiya International', 'Mayank Satapara Surendranagar Gujarat', 'Parva Parmar International Institute']


In [124]:

import math

def compute_semantic_similarity(jd_text, resume_text):
    jd_vec = embedding_model.encode(jd_text, convert_to_tensor=True)
    res_vec = embedding_model.encode(resume_text, convert_to_tensor=True)
    sim = util.cos_sim(jd_vec, res_vec).item()
    return float(max(0.0, sim))

def compute_reranker_score(jd_text, resume_text):
    """Better separation for reranker scores, handling 512 token limit."""
    score = reranker_model.predict([jd_text[:1200], resume_text[:1200]])
    return float(1 / (1 + math.exp(-score / 1.5)))

def compute_evidence_score(matched_skills, exp_text, projects):
    if not matched_skills: return 0.0
    text = (str(exp_text) * 2 + " " + " ".join(map(str, projects))).lower()
    count = sum(1 for skill in matched_skills if normalize_skill(skill) in text)
    return float(count / len(matched_skills))


In [125]:
# Build JD vector
jd_vector = embedding_model.encode(jd_text)

# Build resume vectors (augment with LLM fields + semantic summary)
resume_vectors = []
for resume_text, struct in zip(resume_text_list, Resume_struct):
    # KEPT: semantic summary added to enrichment text
    extra = " ".join(struct["skills"] + struct["education"]) + " " + struct.get("semantic_summary", "")
    full_text = resume_text + " " + extra
    vec = embedding_model.encode(full_text)
    resume_vectors.append(vec)


## Section 6: Skill Matching Score

In [126]:
def semantic_skill_coverage(requirement_skills, candidate_skills):
    """Weighted Skill Scoring: JD-Primary skills (top 8) have 2x authority."""
    if not requirement_skills: return 1.0, [], []
    if not candidate_skills: return 0.0, [], requirement_skills

    req_normalized = [normalize_skill(s) for s in requirement_skills]
    cand_normalized = [normalize_skill(s) for s in candidate_skills]
    cand_set = set(cand_normalized)

    semantic_scores = []
    matched_skills = []
    
    for idx, req_skill in enumerate(requirement_skills):
        weight = 2.0 if idx < 8 else 1.0
        norm_req = req_normalized[idx]
        
        if norm_req in cand_set:
            semantic_scores.append(1.0 * weight)
            matched_skills.append(req_skill)
            continue
            
        req_emb = embedding_model.encode([norm_req], convert_to_tensor=True)
        cand_emb = embedding_model.encode(cand_normalized, convert_to_tensor=True)
        sims = util.cos_sim(req_emb[0], cand_emb)[0].cpu().numpy()
        best_sim = float(np.max(sims)) if len(sims) else 0.0
        
        floor = 0.35
        scaled_sim = max(0, (best_sim - floor) / (1 - floor))
        semantic_scores.append(scaled_sim * weight)
        
        if best_sim > 0.68:
            matched_skills.append(req_skill)

    total_weight = sum(2.0 if i < 8 else 1.0 for i in range(len(requirement_skills)))
    final_score = sum(semantic_scores) / total_weight
    
    missing_skills = [s for s in requirement_skills if s not in matched_skills]
    return float(final_score), matched_skills, missing_skills


## Section 7: Experience Score (LLM-extracted years, no regex)

In [127]:
def experience_score(jd_years, resume_years):
    """Realistic experience scaling for ATS."""
    if jd_years <= 0:
        # If JD asks for 0 (intern/fresher), reward any experience but distinguish 0.2 vs 1.0
        return float(min(0.5 + (resume_years * 0.5), 1.0))
    
    ratio = resume_years / jd_years
    # 0.5 ratio gets 50%, meeting requirement gets 100%, 1.2x gets 110% (bonus)
    score = ratio if ratio < 1.0 else 1.0 + (min(ratio - 1.0, 0.2) * 0.5)
    return float(round(min(score, 1.2), 3))


## Section 8: Project Relevance Score

In [128]:
# ✅ Uses LLM-extracted project titles for scoring

def project_score(jd_text, project_list, model):
    """
    project_list: list of project descriptions from LLM extraction
    """
    if not project_list:
        return 0.0

    jd_vec = model.encode(jd_text, convert_to_tensor=True)
    proj_vecs = model.encode(project_list, convert_to_tensor=True)

    similarities = util.cos_sim(jd_vec, proj_vecs)[0].cpu().numpy()

    best = float(np.max(similarities))
    top_k = sorted(similarities, reverse=True)[:3]
    avg_top = float(np.mean(top_k))

    relevant = [s for s in similarities if s > 0.4]
    relevance_ratio = len(relevant) / len(similarities)

    score = 0.5 * best + 0.3 * avg_top + 0.2 * relevance_ratio
    return round(float(score), 3)


for i, res in enumerate(Resume_struct):
    score = project_score(jd_text, res["projects"], embedding_model)
    print(f"Resume {i+1} Project Score: {score}")

Resume 1 Project Score: 0.23
Resume 2 Project Score: 0.128
Resume 3 Project Score: 0.223
Resume 4 Project Score: 0.233
Resume 5 Project Score: 0.18
Resume 6 Project Score: 0.264
Resume 7 Project Score: 0.189


## Section 9: Education Score

## Section 11: Dynamic JD Weights + Final Scoring

In [129]:
REQUIRED_CUES = [
    "must have", "required", "requirements", "mandatory", "essential",
    "should have", "you should have", "we are looking for"
]
PREFERRED_CUES = [
    "good to have", "nice to have", "preferred", "plus", "bonus"
]

def split_jd_sections(text):
    lower_text = text.lower()
    required_parts, preferred_parts = [], []

    for cue in REQUIRED_CUES:
        idx = lower_text.find(cue)
        if idx != -1:
            required_parts.append(text[idx: idx + 1000])

    for cue in PREFERRED_CUES:
        idx = lower_text.find(cue)
        if idx != -1:
            preferred_parts.append(text[idx: idx + 700])

    return {
        "required_text": "\n".join(required_parts) if required_parts else text,
        "preferred_text": "\n".join(preferred_parts)
    }

def normalize_weights(weights):
    total = sum(weights.values())
    return {k: v / total for k, v in weights.items()} if total > 0 else weights

def parse_jd_requirements(jd_text, jd_struct):
    """Parse JD requirements using LLM-extracted data + ADAPTIVE weight adjustment."""
    sections = split_jd_sections(jd_text)

    all_skills = jd_struct["skills"]
    req_text_lower = sections["required_text"].lower()
    pref_text_lower = sections["preferred_text"].lower()

    required_skills = [s for s in all_skills if s.lower() in req_text_lower] or all_skills[:10]
    preferred_skills = [s for s in all_skills if s.lower() in pref_text_lower
                        and s.lower() not in {r.lower() for r in required_skills}]

    # Base weights
    weights = {
        "required_skills": 0.32,
        "semantic": 0.16,
        "reranker": 0.16,
        "experience": 0.10,
        "education": 0.06,
        "projects": 0.07,
        "evidence": 0.13
    }

    # ADAPTIVE LOGIC
    if any(kw in jd_text.lower() for kw in ["senior", "lead", "years of experience"]):
        weights["experience"] += 0.10
        weights["semantic"] -= 0.10
    
    if jd_struct["experience_years"] == 0:
        weights["required_skills"] += weights["experience"] * 0.5
        weights["projects"] += weights["experience"] * 0.5
        weights["experience"] = 0.0

    if not jd_struct["education"]:
        weights["reranker"] += weights["education"] * 0.5
        weights["semantic"] += weights["education"] * 0.5
        weights["education"] = 0.0

    weights = normalize_weights(weights)

    return {
        "required_skills": required_skills,
        "preferred_skills": preferred_skills,
        "experience_years": jd_struct["experience_years"],
        "education": jd_struct["education"],
        "weights": weights
    }

# --- EXECUTION LINE ---
jd_requirements = parse_jd_requirements(jd_text, JD_struct)

print("Required Skills:", jd_requirements["required_skills"][:10])
print("Preferred Skills:", jd_requirements["preferred_skills"][:5])
print("Experience Req:", jd_requirements["experience_years"])
print("Education Req:", jd_requirements["education"])
print("Weights:", jd_requirements["weights"])


Required Skills: ['C', 'C++', 'Python', 'Java', 'Go', 'Data Structures', 'Algorithms', 'JavaScript', 'TypeScript', 'Linux']
Preferred Skills: []
Experience Req: 0.0
Education Req: ["Master's degree"]
Weights: {'required_skills': 0.37, 'semantic': 0.16, 'reranker': 0.16, 'experience': 0.0, 'education': 0.06, 'projects': 0.12000000000000001, 'evidence': 0.13}


In [130]:
def score_candidate(jd_text, jd_requirements, struct, resume_text, resume_vec):
    req_skill_score, matched_req, missing_req = semantic_skill_coverage(
        jd_requirements["required_skills"], struct["skills"]
    )
    exp_score = experience_score(jd_requirements["experience_years"], struct["experience_years"])
    edu_score = education_score(jd_requirements["education"], struct["education"], resume_text)
    proj_score = project_score(jd_text, struct["projects"], embedding_model)
    
    sem_score = compute_semantic_similarity(jd_text, resume_text)
    
    # Optimized CrossEncoder Usage (Threshold-based)
    if sem_score > 0.35 or req_skill_score > 0.4:
        rerank_score = compute_reranker_score(jd_text, resume_text)
    else:
        rerank_score = sem_score * 0.6  # Scaled down score without expensive compute
        
    ev_score = compute_evidence_score(matched_req, struct["experience_breakdown"], struct["projects"])

    weights = jd_requirements["weights"]
    
    final = 0.0
    if "required_skills" in weights: final += weights['required_skills'] * req_skill_score
    if "semantic" in weights: final += weights['semantic'] * sem_score
    if "reranker" in weights: final += weights['reranker'] * rerank_score
    if "experience" in weights: final += weights['experience'] * exp_score
    if "education" in weights: final += weights['education'] * edu_score
    if "projects" in weights: final += weights['projects'] * proj_score
    if "evidence" in weights: final += weights['evidence'] * ev_score

    return {
        "final_score": round(float(final), 4),
        "required_skill_score": req_skill_score,
        "semantic_score": sem_score,
        "reranker_score": rerank_score,
        "experience_score": exp_score,
        "education_score": edu_score,
        "project_score": proj_score,
        "evidence_score": ev_score,
        "matched_skills": matched_req,
        "missing_skills": missing_req,
        "experience_years": struct["experience_years"]
    }


## Section 12: Candidate Names + Full Ranking Pipeline

In [131]:
def extract_candidate_name_from_resume_text(resume_text):
    """Strict top-line name extraction to avoid grabbing institute names."""
    lines = [l.strip() for l in resume_text.split('\n') if l.strip()]
    if not lines: return None
    
    # Take the first line and limit to first 4 words
    first_line = lines[0]
    words = [w for w in first_line.split() if w.isalpha() and len(w) > 1]
    
    if len(words) >= 2:
        return " ".join(words[:4]).title()
        
    email_match = re.search(r"([a-zA-Z]+)[._]([a-zA-Z]+)@", resume_text)
    if email_match:
        return f"{email_match.group(1).title()} {email_match.group(2).title()}"
    return None


In [132]:
def education_score(jd_edu, resume_edu, resume_text):
    """Cleaned up education matching using normalized keys."""
    if not jd_edu: return 1.0
    
    jd_norms = {normalize_degree(e) for e in jd_edu}
    res_norms = {normalize_degree(e) for e in resume_edu}

    # Direct set intersection
    if jd_norms.intersection(res_norms):
        return 1.0
    
    # Substring check for unmapped degrees
    for req in jd_norms:
        for res in res_norms:
            if req in res or res in req:
                return 1.0
                
    return 0.0


## Section 13: Gap Analysis + Reason Generation

In [133]:
# ============================================================
# 🏆 FULL RANKING PIPELINE
# ============================================================

ranking_results = []

for i, struct in enumerate(Resume_struct):
    try:
        resume_text = resume_text_list[i]
        resume_vec = resume_vectors[i]
        
        # Calculate full ATS scores
        scores = score_candidate(jd_text, jd_requirements, struct, resume_text, resume_vec)
        
        # Use LLM-extracted name if available, otherwise fallback to filename-based name
        scores["name"] = struct.get("name") if struct.get("name") else candidate_names[i]
        
        ranking_results.append(scores)
    except Exception as e:
        print(f"Error scoring candidate {i+1}: {e}")

# Sort by final score
ranking_results = sorted(ranking_results, key=lambda x: x["final_score"], reverse=True)

print("\n" + "=" * 76)
print("FINAL CANDIDATE RANKING")
print("=" * 76)

for rank, r in enumerate(ranking_results, 1):
    print(f"\nRank {rank}: {r['name']} | Overall Score: {r['final_score']:.3f}")
    print("-" * 76)
    print(f"  Required Skills : {r['required_skill_score']:.0%}")
    print(f"  Semantic Fit    : {r['semantic_score']:.2%}")
    print(f"  Reranker Fit    : {r['reranker_score']:.2%}")
    print(f"  Experience      : {r['experience_years']} yrs")
    print(f"  Education       : {r['education_score']:.0%}")
    print(f"  Projects        : {r['project_score']:.0%}")
    print(f"  ✅ Matched : {', '.join(r['matched_skills'][:5])}")

print("\n=== RANKING COMPLETE ===")



FINAL CANDIDATE RANKING

Rank 1: gaurav rajpurohit | Overall Score: 0.362
----------------------------------------------------------------------------
  Required Skills : 52%
  Semantic Fit    : 34.02%
  Reranker Fit    : 0.42%
  Experience      : 0.5 yrs
  Education       : 100%
  Projects        : 22%
  ✅ Matched : C, C++, Python, Java, Data Structures

Rank 2: Mayank Satapara | Overall Score: 0.343
----------------------------------------------------------------------------
  Required Skills : 51%
  Semantic Fit    : 23.11%
  Reranker Fit    : 0.32%
  Experience      : 0.6 yrs
  Education       : 100%
  Projects        : 26%
  ✅ Matched : C, C++, Python, Java, JavaScript

Rank 3: keyur sanjaykumar padiya | Overall Score: 0.327
----------------------------------------------------------------------------
  Required Skills : 43%
  Semantic Fit    : 42.42%
  Reranker Fit    : 0.49%
  Experience      : 0.2 yrs
  Education       : 100%
  Projects        : 18%
  ✅ Matched : C, C++, Python

In [134]:
def generate_explanation(result):
    name = result["name"]
    score = result["final_score"]
    matched = result.get("matched_skills", [])
    missing = result.get("missing_skills", [])
    exp_years = result.get("experience_years", 0)

    top_matched = ", ".join(matched) if matched else "relevant skills"
    top_missing = ", ".join(missing) if missing else ""

    explanation = f"{name} scored {score:.3f}. "
    explanation += f"Demonstrates expertise in {top_matched}, aligning with the JD. "

    if exp_years:
        explanation += f"Has {exp_years:.1f} years of professional experience. "

    if top_missing:
        explanation += f"Gaps identified in: {top_missing}."
    else:
        explanation += "Closely matches all key required skills."

    return explanation

print("\n📝 EXPLANATIONS\n" + "=" * 60)
for r in ranking_results:
    r["explanation"] = generate_explanation(r)
    print(f"\n#{ranking_results.index(r)+1} {r['explanation']}")



📝 EXPLANATIONS

#1 gaurav rajpurohit scored 0.362. Demonstrates expertise in C, C++, Python, aligning with the JD. Has 0.5 years of professional experience. Gaps identified in: Go, TypeScript.

#2 Mayank Satapara scored 0.343. Demonstrates expertise in C, C++, Python, aligning with the JD. Has 0.6 years of professional experience. Gaps identified in: Go, Data Structures.

#3 keyur sanjaykumar padiya scored 0.327. Demonstrates expertise in C, C++, Python, aligning with the JD. Has 0.2 years of professional experience. Gaps identified in: Go, Data Structures.

#4 Jainish Parmar scored 0.312. Demonstrates expertise in C, C++, JavaScript, aligning with the JD. Has 1.0 years of professional experience. Gaps identified in: Python, Java.

#5 Parva Parmar scored 0.308. Demonstrates expertise in C, C++, Python, aligning with the JD. Has 1.0 years of professional experience. Gaps identified in: Go, Data Structures.

#6 kartavya patel scored 0.305. Demonstrates expertise in C, C++, Data Structur